In [ ]:
import numpy as np
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

DATASET_ROOT = Path("../dataset")
POSES_ROOT   = DATASET_ROOT / "poses"
METADATA_DIR = DATASET_ROOT / "metadata"
AUG_ROOT     = DATASET_ROOT / "augmented_tcn"

SIGNS_TARGET = [
    "SOUFFRIR", "AIDER",    "FORT",     "MALADE",   "COEUR",
    "TETE",     "MORT",     "PLEURER",      "NON",   "FROID",
    "MANGER",    "OUI",   "TOMBER", "ACCIDENT", "MARCHER",
    "ENCEINTE", "DORMIR",  "BOIRE",     "CHAUD",  "MEDECIN"
]
SIGN_TO_IDX = {s: i for i, s in enumerate(SIGNS_TARGET)}
NUM_CLASSES  = len(SIGNS_TARGET)
TARGET_T     = 32

TARGET_COUNT          = 1291   # match OUI (majority class)
MAX_VARIANTS_PER_ORIG = 50     # cap per original instance

print(f"Classes: {NUM_CLASSES} | Target per class: {TARGET_COUNT} | Max variants/orig: {MAX_VARIANTS_PER_ORIG}")

In [ ]:
instances = pd.read_csv(DATASET_ROOT / "instances.csv")
face_files = list((POSES_ROOT / "face").glob("*.npy"))
available_ids = {f.stem for f in face_files}

df = instances[instances["id"].isin(available_ids)].copy()
df["label"] = df["sign"].map(SIGN_TO_IDX)
df = df[df["label"].notna()].copy()
df["label"] = df["label"].astype(int)

print(f"Total instances with poses: {len(df)}")
print("\nClass distribution:")
print(df["sign"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(12, 4))
df["sign"].value_counts().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Instance count per sign (raw)")
ax.set_xlabel("Sign")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Remove T=0 instances
lengths = {}
for _, row in df.iterrows():
    arr = np.load(POSES_ROOT / "face" / f"{row['id']}.npy")
    lengths[row["id"]] = arr.shape[0]
df["T"] = df["id"].map(lengths)
df_clean = df[df["T"] > 0].reset_index(drop=True)
print(f"Removed {len(df) - len(df_clean)} instances with T=0")
print(f"Remaining: {len(df_clean)}")

# Load official splits
with open(METADATA_DIR / "splits" / "train.json") as f:
    train_ids = set(json.load(f))
with open(METADATA_DIR / "splits" / "test.json") as f:
    test_ids = set(json.load(f))

df_train = df_clean[df_clean["id"].isin(train_ids)].reset_index(drop=True)
df_test  = df_clean[df_clean["id"].isin(test_ids)].reset_index(drop=True)

print(f"\nTrain: {len(df_train)} | Test: {len(df_test)}")
print("\nTrain class distribution:")
print(df_train["sign"].value_counts().to_string())

In [ ]:
def resample_sequence(arr: np.ndarray, target_T: int) -> np.ndarray:
    """Resample (T, K, 3) array to target_T frames via linear interpolation."""
    T = arr.shape[0]
    if T == 0:
        raise ValueError(f"resample_sequence received empty array (shape {arr.shape})")
    if T == target_T:
        return arr
    idx = np.linspace(0, T - 1, target_T)
    lo  = np.floor(idx).astype(int).clip(0, T - 1)
    hi  = np.ceil(idx).astype(int).clip(0, T - 1)
    a   = (idx - lo)[:, None, None]
    return ((1 - a) * arr[lo] + a * arr[hi]).astype(np.float32)


def load_raw_poses(instance_id: str):
    """Return (body, lhand, rhand) as float32 (T, K, 3) arrays — no resampling."""
    body  = np.load(POSES_ROOT / "pose"       / f"{instance_id}.npy").astype(np.float32)  # MediaPipe full-body pose (33 kpts)
    lhand = np.load(POSES_ROOT / "left_hand"  / f"{instance_id}.npy").astype(np.float32)
    rhand = np.load(POSES_ROOT / "right_hand" / f"{instance_id}.npy").astype(np.float32)
    return body, lhand, rhand


# Sanity check
_sid = df_train["id"].iloc[0]
_b, _l, _r = load_raw_poses(_sid)
assert _b.ndim == 3 and _b.shape[2] == 3, f"Expected (T,K,3), got {_b.shape}"
assert _b.shape[1] == 33, f"Expected 33 body kpts, got {_b.shape[1]}"
assert _l.shape[1] == 21,  f"Expected 21 left-hand kpts, got {_l.shape[1]}"
assert _r.shape[1] == 21,  f"Expected 21 right-hand kpts, got {_r.shape[1]}"
_r32 = resample_sequence(_b, 32)
assert _r32.shape[0] == 32, f"Expected 32 frames, got {_r32.shape[0]}"
assert _r32.shape[1:] == _b.shape[1:], f"Spatial dims changed: {_r32.shape[1:]} vs {_b.shape[1:]}"
print(f"load_raw_poses OK: body={_b.shape}, lhand={_l.shape}, rhand={_r.shape}")
print(f"resample_sequence OK: {_b.shape} -> {_r32.shape}")

In [ ]:
def aug_horizontal_flip(body, lhand, rhand):
    """Flip x-coord (1-x) and swap left/right hands. Source: Benchmarking_Data paper."""
    def flip_x(arr):
        arr = arr.copy()
        arr[:, :, 0] = 1.0 - arr[:, :, 0]
        return arr
    return flip_x(body), flip_x(rhand), flip_x(lhand)  # note: L/R swapped


def aug_temporal_flip(body, lhand, rhand):
    """Reverse frame order."""
    return body[::-1].copy(), lhand[::-1].copy(), rhand[::-1].copy()


def aug_global_translation(body, lhand, rhand, dx: float, dy: float):
    """Shift all keypoints by (dx, dy). Values in [-0.1, 0.1]. Source: Local-global paper."""
    def shift(arr):
        arr = arr.copy()
        arr[:, :, 0] = np.clip(arr[:, :, 0] + dx, 0.0, 1.0)
        arr[:, :, 1] = np.clip(arr[:, :, 1] + dy, 0.0, 1.0)
        return arr
    return shift(body), shift(lhand), shift(rhand)


def aug_global_scaling(body, lhand, rhand, scale: float):
    """Scale all keypoints around joint centroid by factor. Source: handcraft paper."""
    all_x = np.concatenate([body[:, :, 0].ravel(), lhand[:, :, 0].ravel(), rhand[:, :, 0].ravel()])
    all_y = np.concatenate([body[:, :, 1].ravel(), lhand[:, :, 1].ravel(), rhand[:, :, 1].ravel()])
    cx, cy = all_x.mean(), all_y.mean()
    def scale_arr(arr):
        arr = arr.copy()
        arr[:, :, 0] = np.clip((arr[:, :, 0] - cx) * scale + cx, 0.0, 1.0)
        arr[:, :, 1] = np.clip((arr[:, :, 1] - cy) * scale + cy, 0.0, 1.0)
        return arr
    return scale_arr(body), scale_arr(lhand), scale_arr(rhand)


def aug_global_rotation(body, lhand, rhand, angle_deg: float):
    """2D rotation around skeleton centroid. Source: handcraft paper (±5°)."""
    theta = np.radians(angle_deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    all_x = np.concatenate([body[:, :, 0].ravel(), lhand[:, :, 0].ravel(), rhand[:, :, 0].ravel()])
    all_y = np.concatenate([body[:, :, 1].ravel(), lhand[:, :, 1].ravel(), rhand[:, :, 1].ravel()])
    cx, cy = all_x.mean(), all_y.mean()
    def rotate_arr(arr):
        arr = arr.copy()
        x = arr[:, :, 0] - cx
        y = arr[:, :, 1] - cy
        arr[:, :, 0] = np.clip(cos_t * x - sin_t * y + cx, 0.0, 1.0)
        arr[:, :, 1] = np.clip(sin_t * x + cos_t * y + cy, 0.0, 1.0)
        return arr
    return rotate_arr(body), rotate_arr(lhand), rotate_arr(rhand)


def aug_speed_perturbation(body, lhand, rhand, factor: float):
    """Stretch/compress temporal axis by factor before resampling. Source: handcraft paper."""
    T = body.shape[0]
    new_T = max(1, int(round(T * factor)))
    idx = np.linspace(0, T - 1, new_T)
    lo  = np.floor(idx).astype(int).clip(0, T - 1)
    hi  = np.ceil(idx).astype(int).clip(0, T - 1)
    a   = (idx - lo)[:, None, None]
    interp = lambda arr: ((1 - a) * arr[lo] + a * arr[hi]).astype(np.float32)
    return interp(body), interp(lhand), interp(rhand)


# Sanity checks
_b, _l, _r = load_raw_poses(df_train["id"].iloc[0])
_bf, _lf, _rf = aug_horizontal_flip(_b, _l, _r)
assert _bf.shape == _b.shape and np.all(_bf[:, :, 0] == 1.0 - _b[:, :, 0]), "hflip failed"
_bt, _lt, _rt = aug_temporal_flip(_b, _l, _r)
assert np.allclose(_bt[0], _b[-1]), "temporal flip failed"
_bs, _ls, _rs = aug_speed_perturbation(_b, _l, _r, 1.2)
assert _bs.shape[0] == max(1, int(round(_b.shape[0] * 1.2))), "speed perturbation failed"
print("All augmentation functions OK")